In [ ]:
import numpy as np
from typing import *

def quantize_symmetric(data, bit_width=8, per_channel=False, channel_dim=0, eps=1e-8):
    # Qp = 2^(b-1)-1, Qn = -2^(b-1), 例如 Int8: Qp=127, Qn=-128
    Qp = 2**(bit_width-1)-1; Qn = -(2**(bit_width-1))
    if per_channel:
        # 沿指定通道计算 scale, 其他维度规约
        axes = tuple(i for i in range(data.ndim) if i != channel_dim)
        scale = np.max(np.abs(data), axis=axes, keepdims=True) / Qp
    else:
        scale = np.max(np.abs(data)) / Qp
    scale = np.where(scale < eps, 1.0, scale)  # 防除零
    # x_q = clamp(round(x / scale), Qn, Qp)
    q = np.clip(np.round(data/scale), Qn, Qp).astype(np.int8)
    return q, scale

def dequantize_symmetric(q, scale):
    return q.astype(np.float32) * scale

def quantize_asymmetric(data, bit_width=8, per_channel=False, channel_dim=0, eps=1e-8):
    # 非对称: 映射到 [0, 2^b-1], 用 zero_point 对齐浮点 0
    Qmax = 2**bit_width - 1
    if per_channel:
        axes = tuple(i for i in range(data.ndim) if i != channel_dim)
        min_val = np.min(data, axis=axes, keepdims=True)
        max_val = np.max(data, axis=axes, keepdims=True)
    else:
        min_val, max_val = np.min(data), np.max(data)
    scale = (max_val - min_val) / Qmax  # 输入范围 / 量化范围
    scale = np.where(scale < eps, 1.0, scale)
    zp = np.round(-min_val / scale).astype(np.int32)  # 浮点 0 对应的整数偏移
    # x_q = clamp(round(x/scale) + zp, 0, Qmax)
    q = np.clip(np.round(data/scale) + zp, 0, Qmax).astype(np.uint8)
    return q, scale, zp

def dequantize_asymmetric(q, scale, zp):
    return (q.astype(np.float32) - zp) * scale

def quantize_group(data, group_size, bit_width=8, symmetric=True, eps=1e-8):
    # group 量化: 沿 in_channels 分组, 每组独立 scale
    oc, ic = data.shape
    assert ic % group_size == 0
    ng = ic // group_size
    g = data.reshape(oc, ng, group_size)  # (out, n_groups, group_size)
    if symmetric:
        q, s = quantize_symmetric(g, bit_width, per_channel=True, channel_dim=1, eps=eps)
        return q.reshape(oc, -1), s.reshape(oc, ng)
    else:
        q, s, zp = quantize_asymmetric(g, bit_width, per_channel=True, channel_dim=1, eps=eps)
        return q.reshape(oc, -1), s.reshape(oc, ng), zp.reshape(oc, ng)

def dequantize_group(q, scale, group_size, zp=None):
    oc, ic = q.shape; ng = ic // group_size
    qr = q.reshape(oc, ng, group_size)
    sr = scale.reshape(oc, ng, 1)
    if zp is not None:
        return ((qr.astype(np.float32) - zp.reshape(oc, ng, 1)) * sr).reshape(oc, ic)
    return (qr.astype(np.float32) * sr).reshape(oc, ic)

def calibrate_percentile(data, pct=99.9):
    # 百分位校准: 剔除 extremes, 只保留中间 99.9% 的数据范围
    lo = np.percentile(data, 100-pct)
    hi = np.percentile(data, pct)
    return float(lo), float(hi)

def quantize_percentile(data, bit_width=8, pct=99.9, symmetric=True):
    lo, hi = calibrate_percentile(data, pct)
    c = np.clip(data, lo, hi)  # 截断 outlier
    return quantize_symmetric(c, bit_width) if symmetric else quantize_asymmetric(c, bit_width)

def quantization_error(orig, recon):
    # MSE: 均方误差, MAE: 平均绝对误差, SQNR: 信号量化噪声比
    mse = np.mean((orig-recon)**2)
    mae = np.mean(np.abs(orig-recon))
    sp = np.mean(orig**2); np2 = np.mean((orig-recon)**2) + 1e-12
    sqnr = 10 * np.log10(sp / np2)
    return dict(MSE=mse, MAE=mae, SQNR_dB=sqnr, MaxError=np.max(np.abs(orig-recon)))

def quantize_nbit(data, bit_width=4, symmetric=True, per_channel=False, channel_dim=0):
    fn = quantize_symmetric if symmetric else quantize_asymmetric
    return fn(data, bit_width, per_channel, channel_dim)


In [ ]:
if __name__ == '__main__':
    np.random.seed(42)

    print('1. Symmetric quantization Int8')
    x = np.array([0.5, -0.3, 0.8, -0.1, 0.0])
    q, s = quantize_symmetric(x)
    xh = dequantize_symmetric(q, s)
    e = quantization_error(x, xh)
    print(f'  scale={s:.4f}  MSE={e["MSE"]:.2e}  SQNR={e["SQNR_dB"]:.1f}dB')

    print('\n2. Asymmetric quantization Uint8')
    x = np.array([0.2, 0.5, 0.9, 0.0, -0.3])
    q, s, zp = quantize_asymmetric(x)
    xh = dequantize_asymmetric(q, s, zp)
    e = quantization_error(x, xh)
    print(f'  scale={s:.4f}  zp={zp}  SQNR={e["SQNR_dB"]:.1f}dB')

    print('\n3. Per-channel vs per-tensor')
    W = np.random.uniform(-1, 1, (3, 4))
    _, st = quantize_symmetric(W, per_channel=False)
    _, sc = quantize_symmetric(W, per_channel=True)
    print(f'  per-tensor scale: {st:.4f}')
    print(f'  per-channel scale: {np.squeeze(sc)}')

    print('\n4. Group quantization')
    W = np.random.uniform(-1, 1, (2, 8))
    q2, s2 = quantize_group(W, 2, 4); q4, s4 = quantize_group(W, 4, 4)
    e2 = quantization_error(W, dequantize_group(q2, s2, 2))
    e4 = quantization_error(W, dequantize_group(q4, s4, 4))
    print(f'  group=2 SQNR: {e2["SQNR_dB"]:.1f}dB  ({2*4} scales)')
    print(f'  group=4 SQNR: {e4["SQNR_dB"]:.1f}dB  ({2*2} scales)')

    print('\n5. Percentile calibration (anti-outlier)')
    x = np.concatenate([np.random.randn(1000), [100., -200.]])
    _, sm = quantize_symmetric(x); em = quantization_error(x, dequantize_symmetric(*quantize_symmetric(x)))
    _, sp = quantize_percentile(x, pct=99.); ep = quantization_error(x, dequantize_symmetric(*quantize_percentile(x, pct=99.)))
    print(f'  min-max SQNR: {em["SQNR_dB"]:.1f}dB')
    print(f'  pct(99%) SQNR: {ep["SQNR_dB"]:.1f}dB')

    print('\n6. Int4 vs Int8')
    x = np.random.uniform(-1, 1, 10)
    e4 = quantization_error(x, dequantize_symmetric(*quantize_nbit(x, 4, True)))
    e8 = quantization_error(x, dequantize_symmetric(*quantize_nbit(x, 8, True)))
    print(f'  Int4 SQNR: {e4["SQNR_dB"]:.1f}dB   Int8 SQNR: {e8["SQNR_dB"]:.1f}dB')
